#### RAG with PRO Techniques

- RAG with Advanced Techinques

    1. No LangChain! Just native for maximum flexibility

    2. Use LLM to divide the chunks in a sensible way

    3. Use LLM to re-write chunks in a way that's most useful ("document pre-processing)



In [3]:
# imports

import os
from dotenv import load_dotenv
from pathlib import Path
from openai import OpenAI
from pydantic import BaseModel, Field                       # pydantic way of structured outputs
from chromadb import PersistentClient                       # ChromaDB client library
from tqdm import tqdm
import numpy as np
from sklearn.manifold import TSNE                           # projecting chunks to 2D or 3D
import plotly.graph_objects as go

In [4]:
load_dotenv(override=True)

groq_base_url = os.getenv('GROQ_BASE_URL')
groq_api_key = os.getenv('GROQ_API_KEY')

MODEL = 'openai/gpt-oss-120b'
DB_NAME = 'preprocessed_db'
collection_name = 'docs'
embedding_model = 'all-MiniLM-L6-v2'
KNOWLEDGE_BASE_PATH = Path('knowledge-base')
AVERAGE_CHUNK_SIZE = 500

groq = OpenAI(base_url=groq_base_url, api_key=groq_api_key)

In [ ]:
# For RAG friendly result just like LangChain

class Result(BaseModel):
    page_content : str
    metadata : dict

In [8]:
# class to perfectly represent a chunk

class Chunk(BaseModel):
    headline : str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary : str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    orginal_text : str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {'source': document['source'], 'type':document['type']}
        return Result(
            page_content=self.headline + '\n\n' + self.summary + '\n\n' + self.orginal_text, metadata=metadata
        )

In [ ]:
# class to represent the entire data as a list of Chunk

class Chunks(BaseModel):
    chunks : list[Chunk]

#### Three Steps:

1. Fetch documents from the knowledge base, like LangChain did

2. Call an LLM to turn documents into Chunks

3. Store the Chunks in Chroma